# Sequence-to-Sequence Neural Machine Translation (PyTorch + Hugging Face)

A clean, modern implementation of encoder-decoder NMT with PyTorch.

**Architecture progression (PyTorch version):**
1. Basic GRU encoder-decoder
2. (Optional next) Bidirectional encoder
3. (Optional next) Attention mechanism

This notebook follows the TensorFlow notebook structure and uses a modern Hugging Face tokenizer instead of manual vectorization.

## Setup

In [11]:
import random
import zipfile
import urllib.request
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import AutoTokenizer, MarianTokenizer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('mps' if torch.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Torch version: {torch.__version__}')
print(f'GPU available: {torch.mps.is_available() or torch.cuda.is_available()}')
print(f'Device: {device}')

Torch version: 2.11.0
GPU available: True
Device: mps


## Load and Prepare Data

In [12]:
# Download Spanish-English dataset
url = 'https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip'
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
zip_path = data_dir / 'spa-eng.zip'

if not zip_path.exists():
    urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(data_dir)

file_path = next(data_dir.rglob('spa.txt'))
text = file_path.read_text(encoding='utf-8')

# Clean and split into pairs
text = text.replace('¡', '').replace('¿', '')
pairs = [line.split('\t')[:2] for line in text.splitlines() if '\t' in line]
random.shuffle(pairs)

english_sentences, spanish_sentences = zip(*pairs)
print(f'Total sentence pairs: {len(english_sentences):,}')
print('\nExamples:')
for i in range(3):
    print(f'  {english_sentences[i]} => {spanish_sentences[i]}')

Total sentence pairs: 118,964

Examples:
  How long have you been studying Hungarian? => Cuánto tiempo has estado estudiando húngaro?
  Do you really want to be here? => Realmente querés estar acá?
  She is as beautiful as Snow White. => Ella es bella como Blancanieves.


## Text Vectorization

In PyTorch, we often keep tokenization outside the model.
Here we use a modern Hugging Face tokenizer (`AutoTokenizer`) for subword tokenization, truncation, and padding.

In [26]:
# Hyperparameters
MAX_LENGTH = 20
TRAIN_SIZE = 100000
EMBED_DIM = 256
UNITS = 512
BATCH_SIZE = 256

# Modern tokenizer (SentencePiece/BPE-style subword tokenizer)
tokenizer_name = 'Helsinki-NLP/opus-mt-en-es'

try:
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
except Exception as e_auto:
    print('AutoTokenizer failed; trying MarianTokenizer fallback...')
    try:
        tokenizer = MarianTokenizer.from_pretrained(tokenizer_name)
    except Exception as e_marian:
        raise RuntimeError(
            'Tokenizer load failed. Install/upgrade requirements with: '
            'pip install -U transformers sentencepiece protobuf'
        ) from e_marian

# Ensure core special ids exist.
if tokenizer.pad_token_id is None:
    tokenizer.add_special_tokens({'pad_token': '<pad>'})
if tokenizer.eos_token_id is None:
    raise ValueError('Tokenizer has no EOS token; choose a seq2seq-capable tokenizer.')

PAD_ID = tokenizer.pad_token_id
EOS_ID = tokenizer.eos_token_id

# Use a distinct decoder-start token for teacher forcing.
if tokenizer.bos_token_id is not None and tokenizer.bos_token_id not in (PAD_ID, EOS_ID):
    BOS_ID = tokenizer.bos_token_id
else:
    if '<sos>' not in tokenizer.get_vocab():
        tokenizer.add_special_tokens({'additional_special_tokens': ['<sos>']})
    BOS_ID = tokenizer.convert_tokens_to_ids('<sos>')

VOCAB_SIZE = len(tokenizer)

print('Tokenizer:', tokenizer_name)
print('Tokenizer class:', tokenizer.__class__.__name__)
print('Vocab size:', VOCAB_SIZE)
print('PAD_ID:', PAD_ID, 'BOS_ID:', BOS_ID, 'EOS_ID:', EOS_ID)

Tokenizer: Helsinki-NLP/opus-mt-en-es
Tokenizer class: MarianTokenizer
Vocab size: 65002
PAD_ID: 65000 BOS_ID: 65001 EOS_ID: 0


In [27]:
# Prepare datasets
# Encoder input: English sentence
# Decoder input: [BOS] + Spanish tokens
# Target: Spanish tokens + [EOS]

train_eng = list(english_sentences[:TRAIN_SIZE])
train_spa = list(spanish_sentences[:TRAIN_SIZE])
val_eng = list(english_sentences[TRAIN_SIZE:])
val_spa = list(spanish_sentences[TRAIN_SIZE:])

print(f'Training samples: {len(train_eng):,}')
print(f'Validation samples: {len(val_eng):,}')

Training samples: 100,000
Validation samples: 18,964


In [28]:
class NMTDataset(Dataset):
    def __init__(self, src_texts, tgt_texts, tokenizer, max_length=MAX_LENGTH):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src = self.src_texts[idx]
        tgt = self.tgt_texts[idx]

        # Source encoding (English side)
        src_enc = self.tokenizer(
            src,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        # Target encoding (Spanish side): use target-mode tokenization.
        tgt_ids = self.tokenizer(
            text_target=tgt,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )['input_ids'][0]

        # Shift for teacher forcing.
        dec_input = torch.full_like(tgt_ids, PAD_ID)
        dec_input[0] = BOS_ID
        dec_input[1:] = tgt_ids[:-1]

        # Predict the actual target sequence token-by-token.
        dec_target = tgt_ids.clone()

        return {
            'encoder_input_ids': src_enc['input_ids'][0],
            'encoder_attention_mask': src_enc['attention_mask'][0],
            'decoder_input_ids': dec_input,
            'decoder_target_ids': dec_target
        }

train_dataset = NMTDataset(train_eng, train_spa, tokenizer)
val_dataset = NMTDataset(val_eng, val_spa, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
print('Batch shapes:')
for k, v in batch.items():
    print(f'  {k}: {tuple(v.shape)}')

Batch shapes:
  encoder_input_ids: (256, 20)
  encoder_attention_mask: (256, 20)
  decoder_input_ids: (256, 20)
  decoder_target_ids: (256, 20)


## Model 1: Basic Encoder-Decoder

The simplest seq2seq architecture:
- Encoder: GRU reads the source sentence and produces a context vector
- Decoder: GRU generates the target sentence, initialized with the encoder context

In [29]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_size, batch_first=True)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        outputs, hidden = self.gru(x)
        return outputs, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, hidden):
        x = self.embedding(input_ids)
        outputs, hidden = self.gru(x, hidden)
        logits = self.output_layer(outputs)
        return logits, hidden


class BasicSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, pad_idx):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, hidden_size, pad_idx)
        self.decoder = Decoder(vocab_size, embed_dim, hidden_size, pad_idx)

    def forward(self, encoder_input_ids, decoder_input_ids):
        _, hidden = self.encoder(encoder_input_ids)
        logits, _ = self.decoder(decoder_input_ids, hidden)
        return logits


model_basic = BasicSeq2Seq(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_size=UNITS,
    pad_idx=PAD_ID
).to(device)

print(model_basic)

BasicSeq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(65002, 256, padding_idx=65000)
    (gru): GRU(256, 512, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(65002, 256, padding_idx=65000)
    (gru): GRU(256, 512, batch_first=True)
    (output_layer): Linear(in_features=512, out_features=65002, bias=True)
  )
)


In [30]:
# Loss ignores padded positions, equivalent to masking padded targets.
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model_basic.parameters(), lr=1e-3)

def run_epoch(model, dataloader, optimizer=None, max_steps=None, desc=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    total_tokens = 0

    total_steps = len(dataloader) if max_steps is None else min(len(dataloader), max_steps)
    phase = 'train' if training else 'val'
    progress_desc = desc or phase

    progress = tqdm(
        enumerate(dataloader),
        total=total_steps,
        desc=progress_desc,
        leave=False,
        dynamic_ncols=True
    )

    for step, batch in progress:
        if max_steps is not None and step >= max_steps:
            break

        enc_ids = batch['encoder_input_ids'].to(device)
        dec_in_ids = batch['decoder_input_ids'].to(device)
        dec_tgt_ids = batch['decoder_target_ids'].to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(enc_ids, dec_in_ids)
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                dec_tgt_ids.reshape(-1)
            )

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        non_pad = (dec_tgt_ids != PAD_ID).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad

        avg_loss = total_loss / max(total_tokens, 1)
        progress.set_postfix(loss=f'{avg_loss:.4f}', tokens=total_tokens)

    progress.close()
    return total_loss / max(total_tokens, 1)

In [31]:
# Quick sanity training run
EPOCHS = 1

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model_basic, train_loader, optimizer=optimizer)
    val_loss = run_epoch(model_basic, val_loader, optimizer=None, max_steps=20)
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}')

train:   0%|          | 0/391 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Inference: Greedy Decoding

Generate translations one token at a time, always picking the most likely next token.

In [16]:
@torch.no_grad()
def translate_greedy(model, sentence, max_length=MAX_LENGTH):
    model.eval()

    enc = tokenizer(
        sentence,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )

    enc_ids = enc['input_ids'].to(device)

    _, hidden = model.encoder(enc_ids)

    generated = [BOS_ID]
    for _ in range(max_length - 1):
        dec_in = torch.tensor([generated], dtype=torch.long, device=device)
        logits, hidden = model.decoder(dec_in, hidden)
        next_id = int(torch.argmax(logits[:, -1, :], dim=-1).item())

        generated.append(next_id)
        if EOS_ID is not None and next_id == EOS_ID:
            break

    return tokenizer.decode(generated, skip_special_tokens=True)

In [17]:
# Test translations
test_sentences = [
    'I like soccer.',
    'How are you?',
    'The weather is nice today.',
    'Where is the library?'
]

print('=' * 60)
print('Basic Model Translations:')
print('=' * 60)

for sent in test_sentences:
    translation = translate_greedy(model_basic, sent)
    print(f'EN: {sent}')
    print(f'ES: {translation}')
    print()

Basic Model Translations:
EN: I like soccer.
ES: No me que no lo que no lo que no lo que no le.

EN: How are you?
ES: No me de la puerta.

EN: The weather is nice today.
ES: El es un.

EN: Where is the library?
ES: El es un poco.



## Notes for Students

### Key Concepts

1. **Encoder-Decoder Architecture**: Same idea as TensorFlow, implemented with `nn.Module`.
2. **Teacher Forcing**: Decoder input is shifted target tokens.
3. **Modern Vectorization**: Hugging Face tokenizer handles subword tokenization, truncation, and padding.
4. **Masking via Loss**: `ignore_index=PAD_ID` excludes padded target positions from loss.
5. **Explicit Training Loop**: PyTorch gives direct control over forward/backward/optimizer steps.

### Things to Try

- Increase/decrease `UNITS` and `EMBED_DIM`
- Add a bidirectional encoder
- Add additive attention
- Implement beam search decoding
- Add learning rate scheduling